# Trực quan hóa dữ liệu theo ngày bắt đầu (2013-01-02 và phần còn lại)
Notebook này thực hiện việc đọc dữ liệu và vẽ biểu đồ timeline để so sánh 15 mã cổ phiếu lấy được dữ liệu từ `2013-01-02` so với các mã còn lại (có ngày bắt đầu muộn hơn).

In [2]:
import pandas as pd
import os
import glob
import plotly.express as px

# Đọc lại dữ liệu để lấy thông tin ngày bắt đầu
folder_path = 'data/raw/v2' 
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
summary_data = []

for file in csv_files:
    symbol = os.path.basename(file).split('.')[0].upper()
    if len(symbol) <= 4: 
        df = pd.read_csv(file, parse_dates=['Date'])
        if not df.empty:
            summary_data.append({
                'Symbol': symbol,
                'Start Date': df['Date'].min(),
                'End Date': df['Date'].max(),
            })

summary_df = pd.DataFrame(summary_data)

# Phân loại nhóm mã 15 mã từ 2013-01-02 và các mã còn lại
summary_df['Ngày bắt đầu (Chuẩn)'] = summary_df['Start Date'].dt.strftime('%Y-%m-%d')
summary_df['Nhóm'] = summary_df['Ngày bắt đầu (Chuẩn)'].apply(
    lambda x: '15 mã từ 2013-01-02' if x == '2013-01-02' else 'Các mã còn lại'
)

# Sắp xếp theo ngày bắt đầu để biểu đồ đẹp hơn
summary_df = summary_df.sort_values(['Start Date', 'Symbol'])

print("Số lượng theo nhóm:")
print(summary_df['Nhóm'].value_counts())
summary_df.head()

Số lượng theo nhóm:
Nhóm
15 mã từ 2013-01-02    15
Các mã còn lại         15
Name: count, dtype: int64


,Symbol,Start Date,End Date,Ngày bắt đầu (Chuẩn),Nhóm
0,ACB,2013-01-02,2025-12-31,2013-01-02,15 mã từ 2013-01-02
2,BVH,2013-01-02,2025-12-31,2013-01-02,15 mã từ 2013-01-02
3,CTG,2013-01-02,2025-12-31,2013-01-02,15 mã từ 2013-01-02
4,FPT,2013-01-02,2025-12-31,2013-01-02,15 mã từ 2013-01-02
5,GAS,2013-01-02,2025-12-31,2013-01-02,15 mã từ 2013-01-02


### Biểu đồ Timeline trực quan
Sử dụng thư viện Plotly Express để trực quan hóa khoảng thời gian dữ liệu.

In [3]:
# Vẽ biểu đồ timeline trực quan
fig = px.timeline(summary_df, 
                  x_start="Start Date", 
                  x_end="End Date", 
                  y="Symbol", 
                  color="Nhóm",
                  title="Biểu đồ khoảng thời gian dữ liệu của các mã VN30",
                  category_orders={"Symbol": summary_df["Symbol"].tolist()},
                  color_discrete_map={"15 mã từ 2013-01-02": "#1f77b4", "Các mã còn lại": "#ff7f0e"},
                  height=800)

fig.update_yaxes(autorange="reversed")
fig.update_layout(
    xaxis_title="Thời gian",
    yaxis_title="Mã cổ phiếu",
    font=dict(size=12)
)

fig.show()

In [12]:
sector_map = {
    'ACB': ('Ngân hàng',      'Ngân hàng Á Châu'),
    'BVH': ('Bảo hiểm',       'Tập đoàn Bảo Việt'),
    'CTG': ('Ngân hàng',      'VietinBank'),
    'MBB': ('Ngân hàng',      'MB Bank'),
    'STB': ('Ngân hàng',      'Sacombank'),
    'VCB': ('Ngân hàng',      'Vietcombank'),
    'SSI': ('Chứng khoán',    'Chứng khoán SSI'),
    'KDH': ('Bất động sản',   'Khang Điền'),
    'PDR': ('Bất động sản',   'Phát Đạt'),
    'VIC': ('Bất động sản',   'Vingroup'),
    'MSN': ('Tiêu dùng',      'Masan Group'),
    'VNM': ('Tiêu dùng',      'Vinamilk'),
    'HPG': ('Nguyên vật liệu','Hòa Phát Group'),
    'GAS': ('Năng lượng',     'PV Gas'),
    'FPT': ('Công nghệ',      'FPT Corporation'),
}

# Tạo bảng
df = pd.DataFrame([
    {'Ticker': t, 'Company': v[1], 'Sector': v[0]}
    for t, v in sector_map.items()
]).sort_values('Sector')

# Sector weights
weights = df['Sector'].value_counts()
weights_pct = (weights / len(df) * 100).round(1)

print("Phân bổ nhóm ngành:")
for sector, pct in weights_pct.items():
    n   = weights[sector]
    bar = '█' * n
    print(f"  {sector:<15} {n} mã ({pct}%) {bar}")

Phân bổ nhóm ngành:
  Ngân hàng       5 mã (33.3%) █████
  Bất động sản    3 mã (20.0%) ███
  Tiêu dùng       2 mã (13.3%) ██
  Bảo hiểm        1 mã (6.7%) █
  Công nghệ       1 mã (6.7%) █
  Chứng khoán     1 mã (6.7%) █
  Nguyên vật liệu 1 mã (6.7%) █
  Năng lượng      1 mã (6.7%) █


In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Tính toán phân bổ nhóm ngành
sector_counts = df['Sector'].value_counts().reset_index()
sector_counts.columns = ['Sector', 'Count']
sector_counts['Percentage'] = (sector_counts['Count'] / sector_counts['Count'].sum() * 100).round(1)

# Lấy danh sách mã tương ứng với từng nhóm ngành
sector_counts['Tickers'] = sector_counts['Sector'].apply(
    lambda x: ", ".join(df[df['Sector'] == x]['Ticker'].tolist())
)

# Sắp xếp cho biểu đồ cột
bar_data = sector_counts.sort_values('Count', ascending=False)

# Tạo khung chứa 2 biểu đồ (1 hàng, 2 cột)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}]],
    subplot_titles=("Phân bổ Nhóm ngành (%)", "Số lượng cổ phiếu trong từng Nhóm ngành"),
    horizontal_spacing=0.1
)

# 1. Biểu đồ Donut Chart (Nằm bên trái)
fig.add_trace(
    go.Pie(
        labels=sector_counts['Sector'],
        values=sector_counts['Count'],
        hole=0.4,
        customdata=sector_counts[['Tickers', 'Percentage']],
        textinfo='percent+label',
        textposition='inside',
        hovertemplate="<b>%{label}</b><br>Số lượng: %{value} mã (%{customdata[1]}%)<br>Các mã: %{customdata[0]}<extra></extra>"
    ),
    row=1, col=1
)

# 2. Biểu đồ Bar Chart dọc (Nằm bên phải)
fig.add_trace(
    go.Bar(
        x=bar_data['Sector'],
        y=bar_data['Count'],
        text=bar_data['Count'],
        textposition='outside',
        customdata=bar_data[['Tickers']],
        hovertemplate="<b>%{x}</b><br>Số lượng: %{y} mã<br>Các mã: %{customdata[0]}<extra></extra>",
        marker_color=px.colors.qualitative.Plotly[:len(bar_data)]
    ),
    row=1, col=2
)

# Cấu hình layout chung
fig.update_layout(
    title_text="Tổng quan Phân bổ & Số lượng Cổ phiếu (VN30 Sub-set)",
    height=550,
    showlegend=False, 
    font=dict(size=13, family="Arial")
)

# Tăng không gian trục Y một chút để số không bị cắt
fig.update_yaxes(title_text="Số lượng", row=1, col=2, range=[0, bar_data['Count'].max() * 1.2])
fig.update_xaxes(title_text="Nhóm ngành", row=1, col=2)

# Hiển thị biểu đồ
fig.show()